In [1]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/claudeData.xlsx" .

In [2]:
!pip install bert_score sacrebleu rouge-score sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=57dbb21fe77db4cf7da8158be95619a190bf1ef1b46af0ee81eeeed1733dcbaf
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [3]:
!pip install -q "transformers<5" "tokenizers<0.22" "bert-score==0.3.12"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 635.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import pandas as pd
from bert_score import score as bert_score
from sacrebleu.metrics import CHRF
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
df = pd.read_excel("claudeData.xlsx")

In [6]:
ref_col = 'Hop_6_Narrative (Ref)'
pred_col = 'Hop_6_Narrative'

In [7]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('')
df[ref_col] = df[ref_col].fillna('')
print(len(df))

876


## BERTScore-F1

In [8]:
P, R, F1 = bert_score(
    df[pred_col].tolist(),
    df[ref_col].tolist(),
    model_type="HooshvareLab/bert-base-parsbert-uncased",
    num_layers=12,
    lang="fa",
    verbose=True
)

bert_score_f1 = F1.mean().item()

print(f"BERTScore-F1: {bert_score_f1:.4f}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

calculating scores...
computing bert embedding.


  0%|          | 0/25 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/14 [00:00<?, ?it/s]

done in 197.54 seconds, 4.43 sentences/sec
BERTScore-F1: 0.4868


In [9]:
print(f"BERTScore-F1: {bert_score_f1:.4f}")

BERTScore-F1: 0.4868


## chrF++

In [10]:
chrf = CHRF(word_order=2)

chrf_scores = []

for ref, pred in zip(df[ref_col], df[pred_col]):
    score = chrf.sentence_score(pred, [ref])
    chrf_scores.append(score.score)

chrf_score = np.mean(chrf_scores)

print(f"chrF++: {chrf_score:.4f}")

chrF++: 25.7776


## ROUGE-L

In [11]:
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=False
)

rouge_scores = []

for ref, pred in zip(df[ref_col], df[pred_col]):
    score = scorer.score(ref, pred)
    rouge_scores.append(score['rougeL'].fmeasure)

rouge_l = np.mean(rouge_scores)

print(f"ROUGE-L: {rouge_l:.4f}")

ROUGE-L: 0.0000


## Cosine similarity

In [12]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

ref_embeddings = model.encode(
    df[ref_col].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

pred_embeddings = model.encode(
    df[pred_col].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

cosine_scores = []

for ref_embedding, pred_embedding in zip(ref_embeddings, pred_embeddings):
    score = cosine_similarity(
        [ref_embedding],
        [pred_embedding]
    )[0][0]

    cosine_scores.append(score)

cosine_similarity_score = np.mean(cosine_scores)

print(f"Cosine Similarity: {cosine_similarity_score:.4f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Cosine Similarity: 0.6460
